<a href="https://colab.research.google.com/github/SiyuL2025/math_629_final/blob/main/MAT629data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import gc

# ==========================================
# 1. Download and Load Data
# ==========================================
print("1. Downloading dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("martinsn/high-frequency-crypto-limit-order-book-data")
print(f"Dataset downloaded to: {dataset_path}")

# Construct the exact file path for the Bitcoin 1-second data
file_path = os.path.join(dataset_path, 'BTC_1sec.csv')
print(f"Loading data from: {file_path} (reading the first 100,000 rows for testing)...")

df = pd.read_csv(file_path, nrows=100000)
print(f"Data loaded successfully. Original shape: {df.shape}")

# ==========================================
# 2. Feature Engineering
# ==========================================
print("2. Constructing microstructure and market state features...")

# The dataset already provides midpoint and spread
df['mid_price'] = df['midpoint']

# Calculate Order Book Imbalance (OIMB) for the top level (level 0)
b1_v = 'bids_notional_0'
a1_v = 'asks_notional_0'
df['oimb_1'] = (df[b1_v] - df[a1_v]) / (df[b1_v] + df[a1_v] + 1e-8)

# Calculate Market State Variables (Rolling windows: 10s and 60s)
window_short = 10
window_long = 60

df['log_return'] = np.log(df['mid_price'] / df['mid_price'].shift(1))
df['volatility_10s'] = df['log_return'].rolling(window=window_short).std()
df['volatility_60s'] = df['log_return'].rolling(window=window_long).std()
df['momentum_10s'] = df['log_return'].rolling(window=window_short).sum()
df['momentum_60s'] = df['log_return'].rolling(window=window_long).sum()

# Prediction Labels (Predicting price movement for the next 10 seconds)
prediction_horizon = 10
label_threshold = 0.00005

df['future_return'] = (df['mid_price'].shift(-prediction_horizon) - df['mid_price']) / df['mid_price']
df['label'] = np.select(
    [df['future_return'] > label_threshold, df['future_return'] < -label_threshold],
    [1, -1],
    default=0
)

# Drop NaN values generated by rolling windows and shifts
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# ==========================================
# 3. PCA Process
# ==========================================
print("3. Executing PCA dimensionality reduction...")

# Extract the base 60 LOB features: distances and notionals for 15 levels
lob_features = []
for i in range(15):
    lob_features.extend([
        f'bids_distance_{i}', f'asks_distance_{i}',
        f'bids_notional_{i}', f'asks_notional_{i}'
    ])

X_lob = df[lob_features].values

# Train-Test Split (Chronological split: 80% train, 20% test. Do NOT shuffle!)
split_idx = int(len(df) * 0.8)
X_train_lob = X_lob[:split_idx]
X_test_lob = X_lob[split_idx:]

# Standardization (Fit on train, transform on both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_lob)
X_test_scaled = scaler.transform(X_test_lob)

# Apply PCA (Retain components that explain 90% of the variance)
pca = PCA(n_components=0.90, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA completed. Reduced {len(lob_features)} features to {pca.n_components_} components.")

# ==========================================
# 4. Final Dataset Assembly
# ==========================================
print("4. Generating final feature dataset...")

# Convert PCA results back to a DataFrame
pca_cols = [f'pca_{i+1}' for i in range(pca.n_components_)]
pca_df = pd.DataFrame(
    np.vstack((X_train_pca, X_test_pca)),
    columns=pca_cols,
    index=df.index
)

# Select the engineered market state features
market_state_cols = ['spread', 'oimb_1', 'volatility_10s', 'volatility_60s', 'momentum_10s', 'momentum_60s']

# Final X (features) and Y (labels) for deep learning models
X_final = pd.concat([pca_df, df[market_state_cols]], axis=1)
Y_final = df['label']

print(f"Final feature dataset shape: {X_final.shape}")
print(f"Final label dataset shape: {Y_final.shape}")

# Free up memory
gc.collect()

# Display the first few rows
display(X_final.head())

1. Downloading dataset via kagglehub...
Using Colab cache for faster access to the 'high-frequency-crypto-limit-order-book-data' dataset.
Dataset downloaded to: /kaggle/input/high-frequency-crypto-limit-order-book-data
Loading data from: /kaggle/input/high-frequency-crypto-limit-order-book-data/BTC_1sec.csv (reading the first 100,000 rows for testing)...
Data loaded successfully. Original shape: (100000, 156)
2. Constructing microstructure and market state features...
3. Executing PCA dimensionality reduction...
PCA completed. Reduced 60 features to 30 components.
4. Generating final feature dataset...
Final feature dataset shape: (99930, 36)
Final label dataset shape: (99930,)


,pca_1,pca_2,pca_3,pca_4,pca_5,pca_6,pca_7,pca_8,pca_9,pca_10,...,pca_27,pca_28,pca_29,pca_30,spread,oimb_1,volatility_10s,volatility_60s,momentum_10s,momentum_60s
0,-0.239494,3.734093,0.421658,3.610532,-0.263668,2.464975,0.814177,2.189080,3.853346,-0.562814,...,-1.015191,-2.092624,-1.770839,-4.495684,4.28,-0.897520,0.000408,0.000318,-0.000753,-0.002523
1,-1.873595,8.228063,-2.558589,3.775227,4.776096,-0.747831,-0.557820,-3.296721,-1.440594,1.828133,...,0.745981,1.249098,0.653231,0.096348,0.01,-0.993058,0.000397,0.000319,-0.001216,-0.002767
2,5.868840,19.352918,8.996062,2.472510,1.403930,-3.618716,-0.600303,0.609559,-0.811088,3.664230,...,3.362313,1.025563,0.598681,2.200069,25.69,-0.939786,0.000398,0.000320,-0.001528,-0.003075
3,-3.068473,23.332228,8.333070,-0.036848,-1.184740,-2.045731,-0.360462,2.537163,-0.017751,-0.663141,...,1.725265,0.436269,0.195325,-0.572162,25.38,0.447538,0.000325,0.000324,-0.001076,-0.003506
4,-2.942728,15.909479,3.513231,0.629091,-1.180695,-2.370977,-0.678596,2.076109,1.502748,-1.564165,...,0.914723,0.250946,0.202872,0.106557,15.93,0.758075,0.000328,0.000324,-0.001005,-0.003435
